# Bloomberg Terminal — User Behaviour Encoding for Personalization

**Goal:** Transform raw event-stream data (generated by `data_gen/`) into compact, reusable user embeddings that capture behavioural style.

## Pipeline overview

```
CSV events
   └─► Stage 1: Tokenizer      event row → composite token string → int id
   └─► Stage 2: Session Encoder  token sequence → 128-d session embedding (Transformer)
           Pre-train: Masked Event Modeling (MEM)
           Fine-tune: workflow_type classification (22 classes)
   └─► Stage 3: User Encoder   session history → 128-d user embedding (recency attention)
   └─► Stage 4: Evaluation     K-means NMI vs persona, t-SNE visualisation
```

> **Run the data generator first** (if the CSV is missing):
> ```bash
> cd nlp_cores
> python -m data_gen.generate --num-personas 50 --days 30
> ```


In [ ]:
import json
import math
import random
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
from sklearn.metrics import classification_report, normalized_mutual_info_score
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader, Dataset

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Paths & device ────────────────────────────────────────────────────────────
CSV_PATH = Path("data_gen/output/amplitude_events.csv")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"PyTorch {torch.__version__}  |  device: {DEVICE}")
print(f"CSV: {CSV_PATH.resolve()}")


## Stage 0 — Load & Explore Data

In [ ]:
df = pd.read_csv(CSV_PATH)
df["event_properties_parsed"] = df["event_properties"].apply(json.loads)
df["user_properties_parsed"]  = df["user_properties"].apply(json.loads)
df["event_time_dt"] = pd.to_datetime(df["event_time"], utc=True)
df["persona"]        = df["user_properties_parsed"].apply(lambda x: x.get("persona", "unknown"))
df["activity_level"] = df["user_properties_parsed"].apply(lambda x: x.get("activity_level", "unknown"))

print(f"Total events  : {len(df):,}")
print(f"Unique users  : {df['user_id'].nunique()}")
print(f"Unique sessions: {df['session_id'].nunique()}")
print(f"Date range    : {df['event_time_dt'].min()} → {df['event_time_dt'].max()}")
df.head(3)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4))

# Event-type distribution
et_counts = df["event_type"].value_counts()
axes[0].barh(et_counts.index, et_counts.values)
axes[0].set_title("Event-type distribution")
axes[0].set_xlabel("Count")
axes[0].invert_yaxis()

# Workflow distribution (sessions)
wf_counts = df.groupby("session_id")["workflow_type"].first().value_counts()
axes[1].barh(wf_counts.index, wf_counts.values)
axes[1].set_title("Workflow distribution (sessions)")
axes[1].set_xlabel("Sessions")
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

print("Unique workflow types:", df["workflow_type"].nunique())
print("Unique event types   :", df["event_type"].nunique())


## Stage 1 — Tokenizer: Event Row → Composite Token

Each event row is mapped to a **composite token** that bundles:

| Component | Source | Example |
|---|---|---|
| `event_type` | direct column | `document_view` |
| `doc_type` | `event_properties.doc_type` | `case_law` |
| `function_code` | `event_properties.function_code` | `LAW` |
| `jurisdiction` | `event_properties.jurisdiction` | `US_federal` |
| `hour_bucket` | `event_time` (6 buckets) | `morning` |
| `day_type` | `event_time` (weekday/weekend) | `weekday` |

Events missing a property use `*` (wildcard), mapped to a shared index.

**Why composite?** One embedding captures full intent (action + object + context + time)
without needing cross-attention to combine separate embeddings.


In [ ]:
SPECIAL_TOKENS = {'[PAD]': 0, '[CLS]': 1, '[SEP]': 2, '[MASK]': 3}

def _hour_bucket(dt) -> str:
    h = dt.hour
    if h < 6:  return 'night'
    if h < 9:  return 'early'
    if h < 12: return 'morning'
    if h < 15: return 'midday'
    if h < 18: return 'afternoon'
    return 'evening'

def make_composite_token(event_type: str, props: dict, dt) -> str:
    doc_type     = props.get('doc_type', props.get('dataset_type', '*'))
    func_code    = props.get('function_code', '*')
    jurisdiction = props.get('jurisdiction', '*')
    hour_bkt     = _hour_bucket(dt)
    day_type     = 'weekday' if dt.weekday() < 5 else 'weekend'
    return f'{event_type}:{doc_type}:{func_code}:{jurisdiction}:{hour_bkt}:{day_type}'

df['composite_token'] = df.apply(
    lambda r: make_composite_token(
        r['event_type'], r['event_properties_parsed'], r['event_time_dt']
    ),
    axis=1,
)

# Build vocab
vocab = dict(SPECIAL_TOKENS)
for tok in sorted(df['composite_token'].unique()):
    vocab[tok] = len(vocab)

id2tok = {v: k for k, v in vocab.items()}
VOCAB_SIZE = len(vocab)
print(f"Vocabulary size: {VOCAB_SIZE}")
print("\nSample tokens:")
for tok in list(vocab.keys())[4:10]:
    print(f"  {vocab[tok]:3d}  {tok}")


## Stage 2 — Dataset Construction

In [ ]:
# ── Workflow label mapping ─────────────────────────────────────────────────────
workflow_labels   = sorted(df['workflow_type'].unique())
workflow2id       = {w: i for i, w in enumerate(workflow_labels)}
id2workflow       = {i: w for w, i in workflow2id.items()}
NUM_WORKFLOW_CLASSES = len(workflow_labels)
print(f"Workflow classes ({NUM_WORKFLOW_CLASSES}):")
for w in workflow_labels:
    print(f"  {workflow2id[w]:2d}  {w}")


In [ ]:
MAX_SEQ_LEN = 64   # [CLS] + events + [SEP]; longer sessions are truncated

def encode_session(session_df: pd.DataFrame) -> dict:
    session_df = session_df.sort_values('event_time_dt')
    token_ids  = [vocab.get(t, SPECIAL_TOKENS['[MASK]']) for t in session_df['composite_token']]
    timestamps = session_df['event_time_dt'].apply(lambda x: x.timestamp()).tolist()

    # Relative time deltas from session start (seconds)
    t0     = timestamps[0]
    deltas = [t - t0 for t in timestamps]

    # Wrap with special tokens
    ids    = [SPECIAL_TOKENS['[CLS]']] + token_ids  + [SPECIAL_TOKENS['[SEP]']]
    deltas = [0.0]                    + deltas      + [deltas[-1] if deltas else 0.0]

    return {
        'ids'      : ids,
        'deltas'   : deltas,
        'label'    : workflow2id[session_df['workflow_type'].iloc[0]],
        'workflow' : session_df['workflow_type'].iloc[0],
    }

# ── Group into sessions ────────────────────────────────────────────────────────
sessions_by_user: dict[str, list] = defaultdict(list)

for sid, sdf in df.groupby('session_id'):
    uid   = sdf['user_id'].iloc[0]
    enc   = encode_session(sdf)
    sessions_by_user[uid].append({
        **enc,
        'session_id'   : sid,
        'session_start': sdf['event_time_dt'].min(),
        'user_id'      : uid,
        'persona'      : sdf['persona'].iloc[0],
        'activity_level': sdf['activity_level'].iloc[0],
    })

# Sort each user's sessions chronologically
for uid in sessions_by_user:
    sessions_by_user[uid].sort(key=lambda x: x['session_start'])

all_users = sorted(sessions_by_user)
total_sessions = sum(len(v) for v in sessions_by_user.values())
print(f"Users: {len(all_users)} | Sessions: {total_sessions}")
avg_sess = total_sessions / max(len(all_users), 1)
print(f"Avg sessions/user: {avg_sess:.1f}")


In [ ]:
# ── Train / Val / Test split  (always by user, never by event) ────────────────
shuffled = list(all_users)
random.Random(SEED).shuffle(shuffled)
n       = len(shuffled)
n_train = max(1, int(0.70 * n))
n_val   = max(1, int(0.15 * n))

train_users = set(shuffled[:n_train])
val_users   = set(shuffled[n_train : n_train + n_val])
test_users  = set(shuffled[n_train + n_val :])

def flatten(user_set):
    out = []
    for uid in user_set:
        out.extend(sessions_by_user[uid])
    return out

train_sessions = flatten(train_users)
val_sessions   = flatten(val_users)
test_sessions  = flatten(test_users)

print(f"Train: {len(train_users)} users, {len(train_sessions)} sessions")
print(f"Val  : {len(val_users)} users, {len(val_sessions)} sessions")
print(f"Test : {len(test_users)} users, {len(test_sessions)} sessions")


In [ ]:
class SessionDataset(Dataset):
    def __init__(self, sessions: list, max_len: int = MAX_SEQ_LEN):
        self.sessions = sessions
        self.max_len  = max_len

    def __len__(self):
        return len(self.sessions)

    def __getitem__(self, idx):
        s      = self.sessions[idx]
        ids    = s['ids'][:self.max_len]
        deltas = s['deltas'][:self.max_len]

        pad_len         = self.max_len - len(ids)
        attention_mask  = [1] * len(ids) + [0] * pad_len
        ids             = ids    + [SPECIAL_TOKENS['[PAD]']] * pad_len
        deltas          = deltas + [0.0] * pad_len

        return {
            'input_ids'     : torch.tensor(ids,   dtype=torch.long),
            'attention_mask': torch.tensor(attention_mask, dtype=torch.bool),
            'time_deltas'   : torch.tensor(deltas, dtype=torch.float),
            'label'         : torch.tensor(s['label'], dtype=torch.long),
        }

BATCH_SIZE = min(32, max(4, len(train_sessions)))

train_ds = SessionDataset(train_sessions)
val_ds   = SessionDataset(val_sessions)
test_ds  = SessionDataset(test_sessions)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  drop_last=False)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

print(f"Batch size: {BATCH_SIZE} | Train batches/epoch: {len(train_loader)}")


## Stage 3 — Model Architecture

```
Input: [CLS] tok1 tok2 ... tokN [SEP]

Embedding(vocab_size, 128)           — learnable token embedding
TemporalPositionalEncoding           — sinusoidal on log(Δt seconds)
TransformerEncoder(4 layers, 128-d)  — self-attention across events
LayerNorm([CLS] position)            → 128-d session embedding
```

**Pre-training:** Masked Event Modeling (MEM) — mask 15 % of tokens, predict original.
Analogous to BERT MLM, but over the Bloomberg event vocabulary.

**Fine-tuning:** workflow_type classification (22 classes) from the [CLS] embedding.


In [ ]:
D_MODEL    = 128
NHEAD      = 4
NUM_LAYERS = 4

class TemporalPositionalEncoding(nn.Module):
    """Projects log(Δt + 1) seconds into d_model and adds to token embeddings."""
    def __init__(self, d_model: int):
        super().__init__()
        self.proj = nn.Linear(1, d_model)

    def forward(self, x: torch.Tensor, time_deltas: torch.Tensor) -> torch.Tensor:
        log_dt = torch.log1p(time_deltas).unsqueeze(-1)   # (B, L, 1)
        return x + self.proj(log_dt)                       # (B, L, d)


class SessionEncoderCore(nn.Module):
    """Shared transformer trunk used by both pre-training and fine-tuning."""
    def __init__(self, vocab_size: int, d_model: int = D_MODEL,
                 nhead: int = NHEAD, num_layers: int = NUM_LAYERS, dropout: float = 0.1):
        super().__init__()
        self.d_model    = d_model
        self.embedding  = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.temporal   = TemporalPositionalEncoding(d_model)
        enc_layer       = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model * 2,
            dropout=dropout, batch_first=True, norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.norm        = nn.LayerNorm(d_model)

    def forward(self, input_ids, attention_mask, time_deltas):
        x        = self.embedding(input_ids)             # (B, L, d)
        x        = self.temporal(x, time_deltas)         # (B, L, d)
        pad_mask = ~attention_mask                       # True = PAD position
        hidden   = self.transformer(x, src_key_padding_mask=pad_mask)
        hidden   = self.norm(hidden)
        return hidden                                    # (B, L, d)

    def cls(self, input_ids, attention_mask, time_deltas) -> torch.Tensor:
        return self.forward(input_ids, attention_mask, time_deltas)[:, 0, :]  # (B, d)


class MEMModel(nn.Module):
    """Pre-training model: encoder + token-prediction head for MEM."""
    def __init__(self, vocab_size: int):
        super().__init__()
        self.encoder = SessionEncoderCore(vocab_size)
        d = self.encoder.d_model
        self.head = nn.Sequential(
            nn.Linear(d, d), nn.GELU(), nn.LayerNorm(d), nn.Linear(d, vocab_size)
        )

    def forward(self, input_ids, attention_mask, time_deltas):
        hidden = self.encoder(input_ids, attention_mask, time_deltas)  # (B, L, d)
        return self.head(hidden)                                        # (B, L, V)


class WorkflowClassifier(nn.Module):
    """Fine-tuning model: [CLS] embedding → 22-class softmax."""
    def __init__(self, encoder: SessionEncoderCore, num_classes: int, dropout: float = 0.1):
        super().__init__()
        self.encoder = encoder
        d = encoder.d_model
        self.head = nn.Sequential(
            nn.Linear(d, d // 2), nn.GELU(), nn.Dropout(dropout), nn.Linear(d // 2, num_classes)
        )

    def forward(self, input_ids, attention_mask, time_deltas):
        return self.head(self.encoder.cls(input_ids, attention_mask, time_deltas))


# Instantiate
mem_model = MEMModel(vocab_size=VOCAB_SIZE).to(DEVICE)
n_params = sum(p.numel() for p in mem_model.parameters())
print(f"MEM model parameters: {n_params:,}")


## Stage 4 — Pre-training: Masked Event Modeling (MEM)

In [ ]:
def apply_mem_masking(input_ids: torch.Tensor, mask_prob: float = 0.15):
    """Randomly mask 15 % of non-special tokens. Returns (masked_ids, labels)."""
    labels     = input_ids.clone()
    can_mask   = input_ids > 3                # skip PAD, CLS, SEP, MASK
    rand_mask  = torch.bernoulli(
        torch.full(input_ids.shape, mask_prob, device=input_ids.device)
    ).bool()
    mask        = rand_mask & can_mask
    masked_ids  = input_ids.clone()
    masked_ids[mask] = SPECIAL_TOKENS['[MASK]']
    labels[~mask]    = -100                  # CE ignores -100
    return masked_ids, labels


def run_epoch(model, loader, optimizer=None, device=DEVICE):
    training = optimizer is not None
    model.train(training)
    total_loss, n = 0.0, 0
    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for batch in loader:
            ids    = batch['input_ids'].to(device)
            mask   = batch['attention_mask'].to(device)
            deltas = batch['time_deltas'].to(device)

            masked_ids, labels = apply_mem_masking(ids)
            logits = model(masked_ids, mask, deltas)              # (B, L, V)
            loss   = F.cross_entropy(
                logits.reshape(-1, VOCAB_SIZE), labels.reshape(-1), ignore_index=-100
            )
            if training:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            total_loss += loss.item()
            n += 1
    return total_loss / max(n, 1)


In [ ]:
PRETRAIN_EPOCHS = 20
pt_optimizer    = torch.optim.AdamW(mem_model.parameters(), lr=3e-4, weight_decay=0.01)
pt_scheduler    = torch.optim.lr_scheduler.CosineAnnealingLR(pt_optimizer, PRETRAIN_EPOCHS)

pt_train_loss, pt_val_loss = [], []
random_baseline = math.log(VOCAB_SIZE)

print(f"Random-guess MEM loss baseline: {random_baseline:.3f}  (= ln({VOCAB_SIZE}))")
print("Training...")

for epoch in range(1, PRETRAIN_EPOCHS + 1):
    tr = run_epoch(mem_model, train_loader, optimizer=pt_optimizer)
    vl = run_epoch(mem_model, val_loader)
    pt_scheduler.step()
    pt_train_loss.append(tr)
    pt_val_loss.append(vl)
    if epoch % 5 == 0 or epoch == 1:
        print(f"  Epoch {epoch:2d}/{PRETRAIN_EPOCHS}  train={tr:.4f}  val={vl:.4f}")

# ── Loss plot ──────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(pt_train_loss, label='Train')
ax.plot(pt_val_loss,   label='Val', linestyle='--')
ax.axhline(random_baseline, color='gray', linestyle=':', label=f'Random baseline ({random_baseline:.2f})')
ax.set_xlabel('Epoch'); ax.set_ylabel('MEM cross-entropy loss')
ax.set_title('MEM Pre-training Loss'); ax.legend(); plt.tight_layout(); plt.show()
print(f"Final train loss: {pt_train_loss[-1]:.4f} (vs baseline {random_baseline:.2f})")


## Stage 5 — Fine-tuning: Workflow Classification

In [ ]:
def run_finetune_epoch(model, loader, optimizer=None, device=DEVICE):
    training = optimizer is not None
    model.train(training)
    total_loss, correct, total = 0.0, 0, 0
    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for batch in loader:
            ids    = batch['input_ids'].to(device)
            mask   = batch['attention_mask'].to(device)
            deltas = batch['time_deltas'].to(device)
            labels = batch['label'].to(device)

            logits = model(ids, mask, deltas)
            loss   = F.cross_entropy(logits, labels)

            if training:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            total_loss += loss.item()
            correct    += (logits.argmax(-1) == labels).sum().item()
            total      += labels.size(0)
    return total_loss / max(len(loader), 1), correct / max(total, 1)


# Build fine-tuning model from the pre-trained encoder trunk
ft_model = WorkflowClassifier(mem_model.encoder, num_classes=NUM_WORKFLOW_CLASSES).to(DEVICE)
ft_optim = torch.optim.AdamW(ft_model.parameters(), lr=1e-4, weight_decay=0.01)

FINETUNE_EPOCHS  = 25
ft_scheduler     = torch.optim.lr_scheduler.CosineAnnealingLR(ft_optim, FINETUNE_EPOCHS)
majority_baseline = (
    Counter(s['label'] for s in train_sessions).most_common(1)[0][1] / max(len(train_sessions), 1)
)

ft_train_acc, ft_val_acc = [], []
print(f"Majority-class baseline accuracy: {majority_baseline:.3f}")
print("Training...")

for epoch in range(1, FINETUNE_EPOCHS + 1):
    _, tr_acc = run_finetune_epoch(ft_model, train_loader, optimizer=ft_optim)
    _, vl_acc = run_finetune_epoch(ft_model, val_loader)
    ft_scheduler.step()
    ft_train_acc.append(tr_acc)
    ft_val_acc.append(vl_acc)
    if epoch % 5 == 0 or epoch == 1:
        print(f"  Epoch {epoch:2d}/{FINETUNE_EPOCHS}  train_acc={tr_acc:.3f}  val_acc={vl_acc:.3f}")

# ── Accuracy plot ──────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(ft_train_acc, label='Train accuracy')
ax.plot(ft_val_acc,   label='Val accuracy', linestyle='--')
ax.axhline(majority_baseline, color='gray', linestyle=':', label=f'Majority baseline ({majority_baseline:.2f})')
ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
ax.set_title('Workflow Classification Accuracy'); ax.legend(); plt.tight_layout(); plt.show()


In [ ]:
# ── Test set classification report ────────────────────────────────────────────
ft_model.eval()
all_preds, all_labels_list = [], []
with torch.no_grad():
    for batch in test_loader:
        ids    = batch['input_ids'].to(DEVICE)
        mask   = batch['attention_mask'].to(DEVICE)
        deltas = batch['time_deltas'].to(DEVICE)
        preds  = ft_model(ids, mask, deltas).argmax(-1).cpu().tolist()
        all_preds.extend(preds)
        all_labels_list.extend(batch['label'].tolist())

if all_preds:
    print("Test-set classification report:")
    print(classification_report(
        all_labels_list, all_preds,
        target_names=[id2workflow[i] for i in range(NUM_WORKFLOW_CLASSES)],
        zero_division=0,
    ))
else:
    print("No test sessions (too few users). Generate more data with --num-personas 50 --days 30.")


## Stage 6 — User Encoder: Session History → User Embedding

Each user has a variable number of sessions (1 for `inactive`, 100+ for `power_user`).
We aggregate their session embeddings using **recency-weighted attention**:

```
age_days[i] = days before the user's most recent session
decay[i]    = exp(−λ × age_days[i])      λ = 0.05 → 2-week half-life
scores      = softmax(learned_attention + log(decay))
user_emb    = Σ scores[i] × session_emb[i]
```

This gives recent behaviour more weight without discarding older context entirely.


In [ ]:
class RecencyAttentionUserEncoder(nn.Module):
    """Pools session embeddings into a user embedding with recency bias."""
    def __init__(self, d_model: int = D_MODEL, decay_lambda: float = 0.05):
        super().__init__()
        self.attn         = nn.Linear(d_model, 1, bias=False)
        self.decay_lambda = decay_lambda

    def forward(self, session_embs: torch.Tensor, age_days: torch.Tensor) -> torch.Tensor:
        """
        session_embs : (N, d_model)  — ordered oldest → newest
        age_days     : (N,)          — days before most recent session
        """
        decay       = torch.exp(-self.decay_lambda * age_days)          # (N,)
        attn_logits = self.attn(session_embs).squeeze(-1)               # (N,)
        scores      = torch.softmax(attn_logits + torch.log(decay + 1e-8), dim=0)
        return (scores.unsqueeze(-1) * session_embs).sum(0)             # (d_model,)


user_encoder = RecencyAttentionUserEncoder().to(DEVICE)

def get_session_emb(session_record: dict) -> torch.Tensor:
    ds    = SessionDataset([session_record])
    item  = ds[0]
    ids   = item['input_ids'].unsqueeze(0).to(DEVICE)
    mask  = item['attention_mask'].unsqueeze(0).to(DEVICE)
    deltas= item['time_deltas'].unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        return ft_model.encoder.cls(ids, mask, deltas).squeeze(0)   # (d_model,)

def get_user_emb(user_sessions: list) -> torch.Tensor:
    if not user_sessions:
        return torch.zeros(D_MODEL, device=DEVICE)
    latest   = user_sessions[-1]['session_start']
    embs     = torch.stack([get_session_emb(s) for s in user_sessions])
    age_days = torch.tensor(
        [(latest - s['session_start']).total_seconds() / 86400.0 for s in user_sessions],
        dtype=torch.float, device=DEVICE,
    )
    with torch.no_grad():
        return user_encoder(embs, age_days)

# ── Generate embeddings for all users ─────────────────────────────────────────
ft_model.eval(); user_encoder.eval()

records = []
for uid in all_users:
    sessions = sessions_by_user[uid]
    emb = get_user_emb(sessions)
    records.append({
        'user_id'       : uid,
        'persona'       : sessions[0]['persona'],
        'activity_level': sessions[0]['activity_level'],
        'n_sessions'    : len(sessions),
        'embedding'     : emb.cpu().numpy(),
    })

emb_matrix = np.stack([r['embedding'] for r in records])
personas   = [r['persona'] for r in records]
le         = LabelEncoder().fit(personas)
persona_ids= le.transform(personas)

print(f"User embeddings: {emb_matrix.shape}  (users × d_model)")
print(f"Archetypes     : {list(le.classes_)}")


## Stage 7 — Evaluation: Archetype Clustering & Visualisation

In [ ]:
n_archetypes = len(le.classes_)
km = KMeans(n_clusters=n_archetypes, random_state=SEED, n_init=10)
cluster_labels = km.fit_predict(emb_matrix)

nmi = normalized_mutual_info_score(persona_ids, cluster_labels)
print(f"K-means NMI (embeddings vs persona): {nmi:.4f}")
print(f"  0.0 = random baseline  |  1.0 = perfect recovery")

print("\nCluster → dominant persona:")
for c in range(n_archetypes):
    mask_c   = cluster_labels == c
    total_c  = mask_c.sum()
    if total_c == 0:
        continue
    dominant, cnt = Counter(p for p, m in zip(personas, mask_c) if m).most_common(1)[0]
    print(f"  Cluster {c:2d} ({total_c:3d} users): {dominant:<40}  ({cnt}/{total_c} = {cnt/total_c:.0%})")


In [ ]:
perplexity = min(30, max(2, len(records) - 1))
tsne       = TSNE(n_components=2, perplexity=perplexity, random_state=SEED)
coords     = tsne.fit_transform(emb_matrix)

palette = plt.get_cmap('tab10')

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Left: coloured by ground-truth persona
for idx, p in enumerate(le.classes_):
    mask = persona_ids == idx
    axes[0].scatter(coords[mask, 0], coords[mask, 1], color=palette(idx),
                    label=p, alpha=0.85, s=70, edgecolors='white', linewidths=0.3)
axes[0].set_title('User Embeddings — by Persona (ground truth)', fontsize=11)
axes[0].legend(fontsize=7, loc='best', framealpha=0.7)
axes[0].axis('off')

# Right: coloured by K-means cluster
for c in range(n_archetypes):
    mask = cluster_labels == c
    axes[1].scatter(coords[mask, 0], coords[mask, 1], color=palette(c),
                    label=f'Cluster {c}', alpha=0.85, s=70, edgecolors='white', linewidths=0.3)
axes[1].set_title('User Embeddings — by K-means Cluster', fontsize=11)
axes[1].legend(fontsize=7, loc='best', framealpha=0.7)
axes[1].axis('off')

plt.suptitle(
    f't-SNE of 128-d User Behaviour Embeddings  |  NMI={nmi:.3f}',
    fontsize=13, fontweight='bold',
)
plt.tight_layout(); plt.show()
print("The closer the left and right plots look, the more the encoder captured archetypal behaviour.")


## Save User Embeddings

In [ ]:
out_path = Path('data_gen/output/user_embeddings.parquet')
emb_df   = pd.DataFrame({
    'user_id'       : [r['user_id']        for r in records],
    'persona'       : [r['persona']        for r in records],
    'activity_level': [r['activity_level'] for r in records],
    'n_sessions'    : [r['n_sessions']     for r in records],
    **{f'emb_{i}': emb_matrix[:, i] for i in range(D_MODEL)},
})
emb_df.to_parquet(out_path, index=False)
print(f"Saved {len(emb_df)} user embeddings → {out_path}")
print(f"Shape: {emb_df.shape}")
print("\nColumns:", list(emb_df.columns[:8]), '...')
